In [1]:
import torch, transformers
from pathlib import Path
from transformers import BertTokenizer, BertForSequenceClassification

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FINBERT_DIR = PROJECT_ROOT / "models" / "finbert"
print("files:", sorted(p.name for p in FINBERT_DIR.iterdir()))

tokenizer = BertTokenizer.from_pretrained(FINBERT_DIR)
finbert = BertForSequenceClassification.from_pretrained(FINBERT_DIR).eval()

enc = tokenizer("Operating profit fell due to impairment charges .", return_tensors="pt")
with torch.no_grad():
    probs = finbert(**enc).logits.softmax(dim=-1)[0]
print(dict(zip(["negative", "neutral", "positive"], probs.numpy().round(3))))

h:\Desktop\financial-news-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


files: ['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 427.79it/s]


{'negative': np.float32(0.998), 'neutral': np.float32(0.001), 'positive': np.float32(0.001)}


In [5]:
import time, joblib, numpy as np, pandas as pd, keras
from sklearn.metrics import f1_score

LABELS = ["negative", "neutral", "positive"]
train_df = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "train.csv")
test_df  = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test.csv")
texts, y_true = test_df["sentence_clean"].tolist(), test_df["label"].tolist()

# Rebuild the RNN word list exactly like Phase 5
vectorizer = keras.layers.TextVectorization(max_tokens=10000, output_sequence_length=50,
                                            standardize="lower", split="whitespace")
vectorizer.adapt(train_df["sentence_clean"].tolist())

tfidf = joblib.load(PROJECT_ROOT / "models" / "tfidf_vectorizer.joblib")
lr    = joblib.load(PROJECT_ROOT / "models" / "tfidf_lr.joblib")
keras_models = {n: keras.models.load_model(PROJECT_ROOT / "models" / f"{n}.keras")
                for n in ["simple_rnn", "bilstm", "bigru"]}

def predict_one(name, text):
    if name == "tfidf_lr":
        return lr.predict(tfidf.transform([text]))[0]
    if name == "finbert":
        enc = tokenizer(text, truncation=True, max_length=128, return_tensors="pt")
        with torch.no_grad():
            return LABELS[finbert(**enc).logits.argmax(dim=-1).item()]
    probs = keras_models[name](vectorizer([text]), training=False)[0]
    return LABELS[int(np.argmax(probs))]

params = {"tfidf_lr": lr.coef_.size + lr.intercept_.size,
          **{n: m.count_params() for n, m in keras_models.items()},
          "finbert": sum(p.numel() for p in finbert.parameters())}

rows = []
for name in ["tfidf_lr", "simple_rnn", "bilstm", "bigru", "finbert"]:
    for t in texts[:10]:
        predict_one(name, t)                       # warm-up (first calls are always slow)
    times, preds = [], []
    for t in texts:
        t0 = time.perf_counter()
        preds.append(predict_one(name, t))
        times.append((time.perf_counter() - t0) * 1000)
    rows.append({"model": name,
                 "test_macro_f1": round(f1_score(y_true, preds, average="macro"), 4),
                 "params": params[name],
                 "median_ms": round(float(np.median(times)), 2),
                 "p95_ms": round(float(np.percentile(times, 95)), 2)})
    print(rows[-1])

table = pd.DataFrame(rows)
table.to_csv(PROJECT_ROOT / "results" / "comparison.csv", index=False)
table

{'model': 'tfidf_lr', 'test_macro_f1': 0.8059, 'params': 19887, 'median_ms': 0.51, 'p95_ms': 0.67}
{'model': 'simple_rnn', 'test_macro_f1': 0.5776, 'params': 501123, 'median_ms': 93.98, 'p95_ms': 119.3}
{'model': 'bilstm', 'test_macro_f1': 0.7448, 'params': 559107, 'median_ms': 261.42, 'p95_ms': 329.67}
{'model': 'bigru', 'test_macro_f1': 0.7414, 'params': 542979, 'median_ms': 332.27, 'p95_ms': 375.4}
{'model': 'finbert', 'test_macro_f1': 0.9249, 'params': 109754115, 'median_ms': 93.5, 'p95_ms': 136.42}


,model,test_macro_f1,params,median_ms,p95_ms
0,tfidf_lr,0.8059,19887,0.51,0.67
1,simple_rnn,0.5776,501123,93.98,119.30
2,bilstm,0.7448,559107,261.42,329.67
3,bigru,0.7414,542979,332.27,375.40
4,finbert,0.9249,109754115,93.50,136.42


In [6]:
import tensorflow as tf

rows_fast = []
for name, m in keras_models.items():
    fast = tf.function(lambda x, m=m: m(x, training=False))       # compile once
    for t in texts[:10]:
        fast(vectorizer([t]))                                     # warm-up (compiles here)
    times = []
    for t in texts[:200]:
        t0 = time.perf_counter()
        fast(vectorizer([t])).numpy()
        times.append((time.perf_counter() - t0) * 1000)
    rows_fast.append({"model": name, "median_ms_compiled": round(float(np.median(times)), 2)})
    print(rows_fast[-1])

{'model': 'simple_rnn', 'median_ms_compiled': 5.03}
{'model': 'bilstm', 'median_ms_compiled': 5.81}
{'model': 'bigru', 'median_ms_compiled': 5.91}


In [7]:
table = table.merge(pd.DataFrame(rows_fast), on="model", how="left")
table.to_csv(PROJECT_ROOT / "results" / "comparison.csv", index=False)
table

,model,test_macro_f1,params,median_ms,p95_ms,median_ms_compiled
0,tfidf_lr,0.8059,19887,0.51,0.67,NaN
1,simple_rnn,0.5776,501123,93.98,119.30,5.03
2,bilstm,0.7448,559107,261.42,329.67,5.81
3,bigru,0.7414,542979,332.27,375.40,5.91
4,finbert,0.9249,109754115,93.50,136.42,NaN


SyntaxError: invalid syntax (1740787619.py, line 1)